# Predictive Modeling with heterogeneous data

## Overview of scikit-learn library

As introduced before, packages are usually object oriented. Scikit-learn is fully developed around objects which have all the same structures:

* constructor : first definition of models (where you define your parameters)
* fit (method): where you train your model ( iesolve the optimization program) 
* predict (method): use the model to predict a value (for unsupervised model that mean it adds a column with class tag).

Let's have a look at the documentation for the linear regression for instance:

http://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html#sklearn.linear_model.LinearRegression

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings
warnings.simplefilter('ignore', DeprecationWarning)

## Load the dataset with Pandas

We can load the CSV file as a pandas DataFrame in one line:

In [ ]:
data = pd.read_csv('data_train.csv')

In [ ]:
data.head(5)

In [ ]:
data.shape

The DataFrame has 72593 rows and 20 columns.

In [ ]:
list(data.columns)

As it is, the dataframe cannot be directly fed to a scikit-learn model because:


- the target variable (Y) is mixed with the input data

- some attribute such as unique ids (PolNum) have no predictive values for the task

- the values are heterogeneous (string labels for categories, integers and floating point numbers)

Thus, we need to preprocess the data.

## Preprocessing the data

**Step 1** Add a column with whether a policyholder has had an accident.

In [ ]:
def has_encountered_accident(row):
    # [...]
    pass

In [ ]:
data['has_acc'] = data.apply (lambda row: has_encountered_accident (row),axis=1)

Let us have a look at the newly created `has_acc` column:

In [ ]:
Y_column = data['has_acc']
Y_column.dtype

NB: `data['has_acc']` is an instance of the pandas `Series` class with an integer dtype and `data` is an instance of the pandas `DataFrame` class. `Series` can be seen as homogeneous, 1D columns, whereas `DataFrame` instances are heterogenous collections of columns with the same length.

In [ ]:
print(type(Y_column), type(data))

Let's have a quick look at this variable.

In [ ]:
pd.value_counts(data['has_acc'])

In [ ]:
np.mean(data['has_acc'] == 0)

From this the subset of the full policyholder list, about 80% never encountered any accident during their warranty period. So if we are to build a predictive model from this data, a baseline model to compare the performance would be to always predict "no accident". Such a constant model would reach around 82.5% predictive accuracy (which is higher than predicting at random).

**Step 2** Converting to numpy arrays to be understandable by scikit-learn

**`sklearn` estimators all work with homegeneous numerical feature descriptors passed as a numpy array. Therefore passing the raw DataFrame will not work out of the box. You need to convert our DataFrame using the `values`attribute.** 

First, we start with the `Series Y_column`. pandas `Series` instances can be converted to regular 1D numpy arrays by using the `values` attribute:

In [ ]:
target = Y_column.values
type(target) # check

In [ ]:
target.dtype

Then we should convert the features. The method is exactly the same. We preprocess only the features we want to have in the model. To simplify things, let's start with only the numerical features: this way, we don't have to deal with the heterogeneity of the features. 

## Training a predictive model on numerical features

### Preprocessing the features of the model

Let us start simple and build a first model that only uses readily available numerical features as inputs, namely `data.CalYear`, `data.age`, `data.group`, `data.bonus`, `data.poldur`, `data.value` and `data.Density`.

In [ ]:
# extract numerical features
numerical_features = data[['CalYear', 'age', 'group', 'bonus','poldur','value','Density']]
numerical_features.head(5)

We can ensure no missing data can be found within these variables

In [ ]:
numerical_features.count()

We can now convert our Data Frame into an homogeneous numpy array of floating point values:

In [ ]:
# convert to numpy array
features_array = numerical_features.values
features_array # check

In [ ]:
features_array.dtype

We are now nearly ready to run our first modelization. We just need to separate the dataset into two sets: a training set and a test set. Indeed, you should never evaluate a model on the data it was trained on. Otherwise the model may just learn trends of the dataset and not generalize well. 

**Step 3** Split the dataset

Let's take the 80% of the data for training a first model and keep 20% for computing is generalization score. `scikit-learn` has a nice function to do this automatically.

In [ ]:
from sklearn.model_selection import train_test_split

# [...]

In [ ]:
# features_train.shape # 7 features and 80% of the dataset

In [ ]:
# features_test.shape # 7 features and 20 % of the dataset

In [ ]:
# target_train.shape # the Y column and 80 % of the dataset

In [ ]:
# target_test.shape # the Y column and 20 % of the dataset

The preprocessing is finished! Now we can start the modelization. 

### Modeling

Let's start with a simple model from sklearn, namely `LogisticRegression`. If you remember from the first [hands-on](python-part-1.C-hands-on.ipynb), there are three main steps in every modelization with scikit-learn: 
1. Constructor : first definition of models (where you define your parameters)
2. Fit (method): where you train your model ( iesolve the optimization program) 
3. Predict (method): use the model to predict a value (for unsupervised model that mean it adds a column with class tag).


In [ ]:
from sklearn.linear_model import LogisticRegression

# [...]

If you are not sure of how to read the documentation of a function, you can have a look at the documentation of this training material in "./documentation/Tutorial_full_documentation.docx". Let's have a look at the different options for a logistic regression with Scikit-learn.

- `C` parameter indicates the weakness of the regularization (a smaller value >0 stands for a stronger regularzation)


- Use `class_weight` to specify an optionnal weight column to be applied during the regression


- `fit_intercept` if set to `TRUE` allows to add a bias to the decision function


- Use `penallty` parameter to chose between the regular `l2` or `l1` regularization norm


- `solver` lets you indicate which solving method should be used between `{‘newton-cg’, ‘lbfgs’, ‘liblinear’, ‘sag’, ‘saga’}`. Note that for small datasets, `liblinear` is a good choice, whereas `sag` and `saga` are faster for large ones.


- `n_jobs` parameter helps you to speed up your calculations by parallelizing the fitting process according to the number of cores designated by the parameter.

In [ ]:
# [...] # predict

## Model evaluation and interpretation

### A first metric: the accuracy score

Now that we have a model, let's evaluate it. A classic metric is the accuracy score.

In [ ]:
# [...]

In [ ]:
# other method to compute the accuracy score
from sklearn.metrics import accuracy_score
# [...]

This first model has around 83.1% accuracy: this is slightly better than our baseline that always predicts no accident (~ 82.5%).

### Interpreting linear model weights

Often a good accuracy is not enough to convince someone to use a model and we may want to also understand what the model uses to make its predictions: we want to interprete its inner workings. 

The `coef_` attribute of a fitted linear model such as `LogisticRegression` holds the weights of each features.

In [ ]:
# feature_names = numerical_features.columns
# feature_names

In [ ]:
# logreg.coef_

In [ ]:
# x = np.arange(len(feature_names))
# plt.bar(x, logreg.coef_.ravel())
# _ = plt.xticks(x, feature_names, rotation=0)
# _ = plt.title("The weights of the model")
# _ = plt.axhline(0, linestyle="--")

In the model, accidentatlity is positively linked with the group and bonus variables (the higher the bonus, the higher the likelyhood the model will predict accidents for a policyhoder) while older customers are predicted to be less subject to encounter accidents during their warranty period.

### Alternative evaluation metrics

The accuracy metrics aggregates all errors in one measures but it can be interresting to have a closer look at the errors. It is possible to see the details of the false positive and false negative errors by computing the confusion matrix:

In [ ]:
from sklearn.metrics import confusion_matrix

# [...]print(cm)

The true labelings are displayed on rows and the predicted labels on columns:

In [ ]:
def plot_confusion(cm):
    # [...]
    pass
    
plot_confusion(cm)

We can normalize the number of predictions by dividing by the total number of true "survived" and "not survived" to compute false and true positive rates for survival (in the second column of the confusion matrix).

In [ ]:
# [...]

The accuracy score is actually not very informative because the target classes are not balanced in the dataset (~80% 20%). Scikit-learn provides alternative classification metrics to evaluate models performance on imbalanced data such as precision, recall and f1 score:

In [ ]:
from sklearn.metrics import classification_report

# print(classification_report(
#     target_test, 
#     target_predicted,
#     target_names=['no accident', 'had accident(s)'])
# )

A logistic regression is a probabilistic model: it predicts a binary outcome (survived or not) based on the estimated posterior probability of the outcome given the input features. You can access this probability with the `predict_proba` method:

In [ ]:
# target_predicted_proba = logreg.predict_proba(features_test)
# target_predicted_proba[:5]

By default the decision threshold is 0.5 but if you vary the decision threshold from 0 to 1, you can generate a family of binary classifier models that address all the possible trade-offs between false positive and false negative prediction errors.

We can summarize the performance of the binary classifier for all the possible thresholds by plotting the ROC curve and quantifying the Area under the ROC curve:

In [ ]:
from sklearn.metrics import roc_curve
from sklearn.metrics import auc

def plot_roc_curve(target_test, target_predicted_proba):
    # [...]
    pass

In [ ]:
plot_roc_curve(target_test, target_predicted_proba)

Here the area under ROC curve is 0.716 which differs from the accuracy of the model (0.803). Indeed the ROC-AUC score of a random model is expected to 0.5 on average while the accuracy score of a random model depends on the class imbalance of the data. ROC-AUC can be seen as a way to callibrate the predictive accuracy of a model against class imbalance.

### Cross-validation

We previously decided to randomly split the data to evaluate the model on 20% of held-out data. However the randomness of the split might have a significant impact in the estimated accuracy (especially when the orignal dataset is not large). See how the accuracy of the model changes depending on seed (`random_state`) you choose when training it.

In [ ]:
# [...]

In [ ]:
# [...]

In [ ]:
# [...]

So instead of using a single train / test split, we can use a group of them and compute the min, max and mean scores as an estimation of the real test score while not underestimating the variability. This is called **cross-validation**.

In [ ]:
from sklearn.model_selection import cross_val_score

# [...]

In [ ]:
# scores.min(), scores.mean(), scores.max()

`cross_val_score` reports accuracy by default but it can also be used to report other performance metrics such as ROC-AUC or f1-score:

In [ ]:
# [...]

## Exercice 1
- Compute cross-validated scores for other classification metrics ('precision', 'recall', 'f1', 'accuracy'...).

- Change the number of cross-validation folds between 3 and 10: what is the impact on the mean score? on the processing time?

Hints:

The list of classification metrics is available in the online documentation:

  http://scikit-learn.org/stable/modules/model_evaluation.html#common-cases-predefined-values
  
You can use the `%%time` cell magic on the first line of an IPython cell to measure the time of the execution of a cell. 



In [ ]:
# Q1 cross-validated scores for other metrics


In [ ]:
# Q2 number of cross-validation folds


## More feature engineering and richer models

Let's now try to build richer models by including more features as potential predictors for our model.

Categorical variables such as `data.sex`, `data.catgry`, `data.type` and `data.occup` can be converted as boolean indicators features also known as [dummy variables](https://en.wikipedia.org/wiki/Dummy_variable_(statistics)) or one-hot-encoded features:

In [ ]:
pd.get_dummies(data.sex, prefix='sex').head(5)

In [ ]:
pd.get_dummies(data.type, prefix='type').head(5)

We can combine those new numerical features with the previous features using `pandas.concat` along `axis=1`:

In [ ]:
# rich_features =  [...]
# rich_features.head(5)

By construction the new `sex_Male` feature is redundant with `sex_Female`. Let us drop it:

In [ ]:
# rich_features_no_male = rich_features.drop(['sex_Male'], 1)
# rich_features_no_male.head(5)

We can finally cross-validate a logistic regression model on this new data :

In [ ]:
%%time

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# [...]
print("Logistic Regression CV scores:")
print("min: {:.3f}, mean: {:.3f}, max: {:.3f}".format(
    scores.min(), scores.mean(), scores.max()))

## Exercise 2

- change the value of the parameter `C`. Does it have an impact on the score?

- fit a new instance of the logistic regression model on the full dataset.

- plot the weights for the features of this newly fitted logistic regression model.



**Be careful** 

$C$ in scikit learn is a factor of the likelihood functions (whereas in a traditional Ridge/Lasso optimisation where it is usual to find it associated to the weigths norm). Thus $C$ has to be in $]0,+\infty[$

The program optimized is :

$$ ||w||_{1~or~2} + C\sum_{i}^nlog(exp(-y_i(X_i^Tw+c))+1)$$

#### Remark to preprocess your data

You can build your own pipeline with preprocessing class of scikit-learn http://scikit-learn.org/stable/modules/preprocessing.html 

### Training Non-linear models: ensembles of randomized trees

We can train now with some non linear models. Let us consider the example of a decision tree.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
%%time

# [...]
print("DecisionTreeClassifier CV scores:")
print("min: {:.3f}, mean: {:.3f}, max: {:.3f}".format(
    scores.min(), scores.mean(), scores.max()))

In [ ]:
# [...]

`sklearn` also implements non linear models that are known to perform very well for data-science projects where datasets have not too many features (e.g. less than 5000).

In particular let us have a look at Random Forests and Gradient Boosted Trees:

In [ ]:
%%time

from sklearn.ensemble import RandomForestClassifier

# [...]
print("Random Forest CV scores:")
print("min: {:.3f}, mean: {:.3f}, max: {:.3f}".format(
    scores.min(), scores.mean(), scores.max()))

Some useful parameters for the RandomForestClassifier method within scikit-learn are:

- `n_estimators` to specify the number of trees to be built


- `criterion` to chose between `gini` impurity and `entropy` for the information gain


- `max_depth` to define the maximum depth of the trees


- `min_samples_split/min_samples_leaf ` : to determine the minimum number of samples required to split an internal node/be at a leaf node

In [ ]:
%%time

from sklearn.ensemble import GradientBoostingClassifier

# [...]
print("Gradient Boosted Trees CV scores:")
print("min: {:.3f}, mean: {:.3f}, max: {:.3f}".format(
    scores.min(), scores.mean(), scores.max()))

Some useful parameters for the GradientBoostingClassifier method within scikit-learn are:

- `loss` to choose which lost function should be optimized within `{‘deviance’, ‘exponential’}`.


- `learning_rate` to shrink the contribution of each tree by learning_rate. There is a trade-off between learning_rate and n_estimators.


- `criterion` to define the function to measure the quality of a split, to be selected between `{friedman_mse, mse, mae}`


- `n_estimators ` : The number of boosting stages to perform. Gradient boosting is fairly robust to over-fitting so a large number usually results in better performance.


- `subsample` : The fraction of samples to be used for fitting the individual base learners. If smaller than 1.0 this results in Stochastic Gradient Boosting. subsample interacts with the parameter n_estimators. Choosing subsample < 1.0 leads to a reduction of variance and an increase in bias.



Previously defined parameters from RandomForestClassifiers also apply here such as `min_samples_split/min_samples_leaf `.

Both models seem to perform as the logistic regression model on this data.

## Exercise 3

1. Change the value of the `learning_rate` and other `GradientBoostingClassifier` parameters, can you get a better mean score?

2. Would treating the `group` variable as categorical improve the models performance?

3. Find out which predictor variables (features) are the most informative for those models.

Hints:

Fitted ensembles of trees have `feature_importances_` attribute that can be used similarly to the `coef_` attribute of linear models.



In [ ]:
# Q1.

In [ ]:
# Q2.

In [ ]:
# Q3.

In [ ]:
# plot

## Automated parameter tuning

Instead of changing the value of the learning rate manually and re-running the cross-validation, we can find the best values for the parameters automatically (assuming we are ready to wait):

In [ ]:
%%time

from sklearn.model_selection import GridSearchCV

gb = GradientBoostingClassifier(n_estimators=100, subsample=.8)

params = {
    'learning_rate': [0.05, 0.1, 0.5],
    'max_features': [0.5, 1],
    'max_depth': [3, 4, 5],
}
gs = GridSearchCV(gb, params, cv=5, scoring='accuracy', n_jobs=4)
gs.fit(rich_features_no_male, target)

Let us sort the models by mean validation score:

In [ ]:
gs.cv_results_['mean_test_score']

In [ ]:
df_cv_results = pd.DataFrame.from_dict(gs.cv_results_)
df_cv_results.sort_values('mean_test_score', ascending=False)

In [ ]:
gs.best_score_ # see the best score

In [ ]:
gs.best_params_ # see the best trio of parameters

We should note that the mean scores are very close to one another and almost always within one standard deviation of one another. This means that all those parameters are quite reasonable.

## Exercise 4 (optional)

1. Load the test dataset located at '../materials/data_test.csv'
2. Apply same transformation as you did on your train data (dummies)
3. Predict the target on the test dataset with the best Gradient Boosting Classifier. Compute its accuracy using Sklearn score function. Did the model overfit ?

In [ ]:
# Correction
# Q1. Load data_test.csv


In [ ]:
# Q2. Prepare X

# Prepare y

In [ ]:
# Q3.

### Saving models

Most models have a save and a load method. It saves the model into a python format with the fitted parameters. In that way, in a production process, you can load a model you fitted on a specific sample and use it to make predictions on another sample.

**Note**

Most Python objects can be saved into a textfile format. The package you use for this is called **pickle**. For an in-depth presentation of Pickle, see the [notebook 1B on pickle](python-part-1.B-pickle-facultatif.ipynb). You might see .pickle files sometimes, it means it is a python object that is saved in the file. We present below how to load such a pickle file.

The method is very close for Statmodels and Sklearn.

#### statsmodels

In [ ]:
results.save(fname='../outputs/GLM')

In [ ]:
# you can then load the pickle
from statsmodels.iolib.smpickle import load_pickle
new_results = load_pickle("../outputs/GLM")
new_results.summary2()

#### scikit-learn

In [ ]:
#save
import pickle
s = pickle.dumps(logreg)
l = open("../outputs/mylogreg.pickle",'wb')
l.write(s)
l.close()

In [ ]:
#load
import pickle
l = open("../outputs/mylogreg.pickle",'rb')
mypickle = l.read()
l.close()
logreg = pickle.loads(mypickle)